# Notebook 05 — Grad-CAM Explainability

**Phase 6 of the Wafer Defect Detection & Yield Risk Dashboard**

Goal: Apply Grad-CAM to the trained WaferCNN to understand what spatial regions drive predictions.

We focus on:
- All 9 defect classes (correct predictions)
- Hard classes: Scratch (F1=0.048) and Edge-Loc (F1=0.376) with misclassifications
- A summary heatmap grid saved to `outputs/figures/gradcam_examples.png`

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

from src.explainability import (
    GradCAM, predict_single, overlay_heatmap,
    plot_gradcam_grid, collect_gradcam_samples, load_model
)
from src.preprocessing import load_processed, PROCESSED_DIR
from src.data_loader import DEFECT_CLASSES

FIGURES_DIR = Path('../outputs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

## 1. Load model and test data

In [ ]:
MODEL_PATH = Path('../models/best_model.pth')

model = load_model(MODEL_PATH)
print(f'Model loaded from {MODEL_PATH}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

# Load test split only — we never touch test data during training
X_train, X_val, X_test, y_train, y_val, y_test = load_processed(PROCESSED_DIR)
print(f'Test set: {X_test.shape[0]:,} samples')

## 2. Quick sanity check — single Grad-CAM

In [ ]:
# Pick the first test sample and run Grad-CAM
idx = 0
x_np = X_test[idx]                                # (1, 64, 64)
x_t  = torch.from_numpy(x_np).unsqueeze(0)        # (1, 1, 64, 64)

grad_cam = GradCAM(model)
pred_idx, confidence, probs = predict_single(model, x_t)
heatmap = grad_cam(x_t, class_idx=pred_idx)
grad_cam.remove_hooks()

true_label = DEFECT_CLASSES[y_test[idx]]
pred_label = DEFECT_CLASSES[pred_idx]

print(f'True: {true_label}  |  Pred: {pred_label}  |  Confidence: {confidence:.1%}')
print(f'Heatmap range: [{heatmap.min():.3f}, {heatmap.max():.3f}]')

overlay = overlay_heatmap(x_np[0], heatmap)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(x_np[0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
axes[0].set_title(f'Wafer Map\nTrue: {true_label}', fontsize=10)
axes[0].axis('off')
axes[1].imshow(overlay, interpolation='nearest')
axes[1].set_title(f'Grad-CAM Overlay\nPred: {pred_label} ({confidence:.0%})', fontsize=10)
axes[1].axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gradcam_sanity.png', dpi=120)
plt.show()
print('Sanity check passed')

## 3. Correct predictions — one per class

This shows what spatial regions the model correctly learned to focus on for each defect pattern.

In [ ]:
print('Collecting correct-prediction Grad-CAM samples (1 per class)...')
correct_samples = collect_gradcam_samples(
    model, X_test, y_test,
    n_per_class=1,
    correct_only=True,
    seed=42,
)
print(f'Collected {len(correct_samples)} samples')
for s in correct_samples:
    print(f"  {s['true_label']:<15} pred={s['pred_label']:<15} conf={s['confidence']:.1%}")

In [ ]:
plot_gradcam_grid(
    correct_samples,
    save_path=FIGURES_DIR / 'gradcam_correct.png',
    title='Grad-CAM — Correctly Classified Samples (1 per class)',
)
print('Saved: outputs/figures/gradcam_correct.png')

## 4. Misclassifications — focus on hard classes

Scratch (F1=0.048) and Edge-Loc (F1=0.376) were flagged as the two weakest classes. 
We look at misclassified examples to understand where the model goes wrong.

In [ ]:
print('Collecting misclassification Grad-CAM samples...')
incorrect_samples = collect_gradcam_samples(
    model, X_test, y_test,
    n_per_class=2,
    incorrect_only=True,
    seed=42,
)
print(f'Collected {len(incorrect_samples)} misclassified samples')
for s in incorrect_samples:
    print(f"  True: {s['true_label']:<15} Pred: {s['pred_label']:<15} conf={s['confidence']:.1%}")

In [ ]:
plot_gradcam_grid(
    incorrect_samples,
    save_path=FIGURES_DIR / 'gradcam_incorrect.png',
    title='Grad-CAM — Misclassified Samples',
)
print('Saved: outputs/figures/gradcam_incorrect.png')

## 5. Deep dive — Scratch class

Scratch has F1=0.048, the lowest of all classes. The defect is a thin diagonal or curved line across the wafer. 
We look at both correct and incorrect predictions to understand what makes it hard.

In [ ]:
SCRATCH_IDX = DEFECT_CLASSES.index('Scratch')
scratch_indices = np.where(y_test == SCRATCH_IDX)[0]
print(f'Scratch samples in test set: {len(scratch_indices)}')

# Collect up to 6 scratch samples (mix of correct and incorrect)
scratch_samples = collect_gradcam_samples(
    model, X_test, y_test,
    n_per_class=6,
    seed=42,
)
scratch_samples = [s for s in scratch_samples if s['true_label'] == 'Scratch']
print(f'Collected {len(scratch_samples)} Scratch samples')

if scratch_samples:
    plot_gradcam_grid(
        scratch_samples,
        save_path=FIGURES_DIR / 'gradcam_scratch_deep_dive.png',
        title='Grad-CAM Deep Dive — Scratch Class',
    )
    print('Saved: outputs/figures/gradcam_scratch_deep_dive.png')

## 6. Main portfolio figure — all-class Grad-CAM grid

This is the primary output figure for the README and dashboard. 
Two samples per class: one correct, one incorrect (where available).

In [ ]:
print('Building main Grad-CAM portfolio figure...')
portfolio_samples = collect_gradcam_samples(
    model, X_test, y_test,
    n_per_class=1,
    correct_only=False,
    seed=7,
)
print(f'Collected {len(portfolio_samples)} samples')

plot_gradcam_grid(
    portfolio_samples,
    save_path=FIGURES_DIR / 'gradcam_examples.png',
    title='Grad-CAM — WaferCNN Spatial Explanations',
)
print('Saved: outputs/figures/gradcam_examples.png')

## 7. Class-level attention analysis

For each class, compute the average heatmap over multiple test samples. 
This gives a stable picture of where the model typically looks for each defect pattern.

In [ ]:
print('Computing average Grad-CAM heatmaps per class...')

rng = np.random.default_rng(42)
N_AVG = 20  # samples per class for averaging

grad_cam = GradCAM(model)
avg_heatmaps = {}

for class_idx, class_name in enumerate(DEFECT_CLASSES):
    indices = np.where(y_test == class_idx)[0]
    if len(indices) == 0:
        continue
    chosen = rng.choice(indices, size=min(N_AVG, len(indices)), replace=False)
    
    maps = []
    for idx in chosen:
        x_np = X_test[idx]
        x_t  = torch.from_numpy(x_np).unsqueeze(0)
        h    = grad_cam(x_t, class_idx=class_idx)  # force target class
        maps.append(h)
    
    avg_heatmaps[class_name] = np.stack(maps).mean(axis=0)
    print(f'  {class_name:<15}: {len(chosen)} samples averaged')

grad_cam.remove_hooks()
print('Done')

In [ ]:
n_classes = len(avg_heatmaps)
fig, axes = plt.subplots(1, n_classes, figsize=(2.5 * n_classes, 3))
fig.suptitle('Average Grad-CAM Heatmap per Defect Class', fontsize=12, fontweight='bold')

for ax, (class_name, heatmap) in zip(axes, avg_heatmaps.items()):
    ax.imshow(heatmap, cmap='jet', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(class_name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gradcam_avg_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/gradcam_avg_per_class.png')

## 8. Summary

Run this cell to print a summary of outputs produced.

In [ ]:
outputs = [
    'gradcam_sanity.png',
    'gradcam_correct.png',
    'gradcam_incorrect.png',
    'gradcam_scratch_deep_dive.png',
    'gradcam_examples.png',
    'gradcam_avg_per_class.png',
]

print('Phase 6 — Grad-CAM Outputs:')
for f in outputs:
    path = FIGURES_DIR / f
    status = 'OK' if path.exists() else 'MISSING'
    size_kb = f'{path.stat().st_size / 1024:.0f} KB' if path.exists() else '—'
    print(f'  [{status}] {f} ({size_kb})')

print()
print('Phase 6 complete.')
print('Next: Phase 7 — Risk Scoring (src/risk_scoring.py)')